In [1]:
import numpy as np
import pandas as pd

Load raw_trades.csv, then report its shape, dtypes and null counts in three lines. 

In [2]:
# path is relative to this notebook, so it works on any clone of the repo
raw_trades = pd.read_csv('../data/raw_trades.csv')

In [3]:
raw_trades.head()

,trade_id,ticker,trade_date,price,volume,venue_code,side
0,1044,JNJ,2026-01-03,NaN,1000.0,XNYS,SELL
1,1045,JNJ,2026-01-03,NaN,7500.0,XNYS,SELL
2,1012,MSFT,2026-01-07,438.92,1500.0,XNAS,SELL
3,1002,AAPL,2026-01-08,224.96,1000.0,XNAS,BUY
4,1013,MSFT,2026-01-11,447.60,7500.0,XNAS,BUY


In [4]:
#show the shape of the dataframe (rows, columns)
raw_trades.shape

(56, 7)

In [5]:
#inspect the different data types
raw_trades.dtypes

trade_id        int64
ticker            str
trade_date        str
price         float64
volume        float64
venue_code        str
side              str
dtype: object

In [6]:
#count how many rows are null per column
raw_trades.isna().sum()

trade_id      0
ticker        0
trade_date    0
price         6
volume        3
venue_code    0
side          0
dtype: int64

Set trade_date as the index and select January's trades two ways — with .loc and with .iloc — and write one line on when each is appropriate.

In [7]:
#set trade date as index
raw_trades.set_index("trade_date", inplace=True)

In [8]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY


In [9]:
raw_trades.sort_index(ascending=True, inplace=True)

In [10]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY


In [11]:
#select january trades
#start and end
raw_trades.loc['2026-01-01':'2026-01-31']

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY


In [12]:
#positional
raw_trades.iloc[:9]

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY


**`.loc` vs `.iloc`:** use `.loc` when you mean the data itself — a date range like January — because labels survive sorting, filtering and row drops; use `.iloc` when you genuinely mean position rather than content, such as taking the top 5 after a deliberate sort, and accept that those positions shift the moment the frame changes.

Which column has the wrong dtype straight off the CSV, and why did that happen?

**`trade_date` comes in as a string when it should be a datetime.** A CSV carries no type information at all — every value in the file is plain text. pandas infers types on read, and it can safely infer numbers, but dates are ambiguous (is `01/02/2026` 1 February or 2 January?), so it refuses to guess and leaves them as strings unless told explicitly via `parse_dates`. In BigQuery `trade_date` is declared `DATE` in the schema, so the type travels with the data; exporting to CSV throws that away.

**Also worth noting: `volume` reads back as `float64`, not an integer**, despite share volumes being whole numbers. `NaN` is a float value and NumPy's `int64` has no representation for "missing", so a single null forces the entire column to upcast to float. Once the nulls are handled on Tuesday this can be converted back to `int`, or to pandas' nullable `Int64` which holds missing values while staying integer.

Detect the nulls: 6 in price, 3 in volume. Show that df.price == None finds none of them and explain why.

In [13]:
raw_trades.isna().sum()

trade_id      0
ticker        0
price         6
volume        3
venue_code    0
side          0
dtype: int64

In [14]:
# .isna() finds the nulls; equality comparison finds none of them
print('found by .isna():   ', raw_trades.price.isna().sum())
print('found by == None:   ', (raw_trades.price == None).sum())
print('found by == np.nan: ', (raw_trades.price == np.nan).sum())

# why: NaN is not equal to anything, including itself
print()
print('np.nan == np.nan ->', np.nan == np.nan)
print('None == None     ->', None == None)

found by .isna():    6
found by == None:    0
found by == np.nan:  0

np.nan == np.nan -> False
None == None     -> True


**Why `== None` finds nothing.** Two separate reasons stack up:

1. **The missing values aren't `None`.** In a numeric column pandas stores missing as `NaN`, a float from the IEEE 754 spec — a different object entirely from Python's `None`. Comparing against `None` asks the wrong question.

2. **`NaN` isn't equal to anything, including itself.** IEEE 754 defines it that way: `np.nan == np.nan` is `False`. NaN means "undefined", and an undefined value can't be shown equal to anything — so even `== np.nan` returns all `False`.

The result is a mask that's `False` for all 56 rows, selecting nothing. `.isna()` exists precisely because equality can't do this job — it tests *is this value missing* rather than *does this value equal that one*.

**Contrast with SQL's `= NULL` (Week 8):** same practical outcome, different mechanism. SQL uses three-valued logic — `price = NULL` evaluates to `NULL` (unknown), and `WHERE` keeps only rows that are `TRUE`. pandas has no third value: the comparison returns plain `False`. Either way you need a purpose-built test — `IS NULL` in SQL, `.isna()` here.

In [15]:
#drop duplicates
raw_trades.drop_duplicates(inplace=True)

In [16]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY


In [17]:
raw_trades.shape

(54, 6)

In [18]:
raw_trades['price'] = raw_trades['price'].fillna(raw_trades.groupby(['ticker'])['price'].transform('mean'))

In [19]:
raw_trades.isna().sum()

trade_id      0
ticker        0
price         0
volume        3
venue_code    0
side          0
dtype: int64

In [20]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,152.345556,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,152.345556,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.920000,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.960000,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.600000,7500.0,XNAS,BUY


### Comparing a per-instrument fill against a global mean

Rebuilt from source below so both strategies start from the same 54 rows with the original 6 null prices intact. Overwriting the only copy of a column destroys the ability to compare alternatives, so each strategy is kept as its own named frame instead.

In [21]:
raw = (pd.read_csv('../data/raw_trades.csv')
         .set_index('trade_date')
         .sort_index()
         .drop_duplicates())

per_instrument = raw.assign(price=raw['price'].fillna(raw.groupby('ticker')['price'].transform('mean')))
global_mean    = raw.assign(price=raw['price'].fillna(raw['price'].mean()))

tickers = sorted(raw['ticker'].unique())
comparison = pd.DataFrame({
    'null_prices': raw[raw['price'].isna()]['ticker'].value_counts().reindex(tickers, fill_value=0),
    'per_instrument_fill': per_instrument.groupby('ticker')['price'].mean().round(2),
    'global_fill': global_mean.groupby('ticker')['price'].mean().round(2),
})
comparison['gap'] = (comparison['global_fill'] - comparison['per_instrument_fill']).round(2)

print(f"overall mean price, used by the global fill: {raw['price'].mean():.2f}\n")
comparison

overall mean price, used by the global fill: 226.46



,null_prices,per_instrument_fill,global_fill,gap
ticker,,,,
AAPL,1,221.88,222.30,0.42
JNJ,2,152.35,165.82,13.47
JPM,0,214.34,214.34,0.00
MSFT,3,468.00,402.12,-65.88
XOM,0,117.82,117.82,0.00


**Result.** The overall mean price across all five instruments is **£226.46**, and that single number is what a global fill would insert into every gap.

| ticker | null prices | per-instrument fill | global fill | gap |
|---|---|---|---|---|
| AAPL | 1 | 221.88 | 222.30 | +0.42 |
| JNJ | 2 | 152.35 | 165.82 | +13.47 |
| JPM | 0 | 214.34 | 214.34 | 0.00 |
| MSFT | 3 | 468.00 | 402.12 | **−65.88** |
| XOM | 0 | 117.82 | 117.82 | 0.00 |

**AAPL's gap is only £0.42** — much smaller than expected, and the reason is instructive rather than disappointing: AAPL caught just one of the six nulls, and it happens to trade at ~£222, almost exactly the overall mean of £226.46. A global fill inserts roughly the right number by luck. AAPL is the instrument this error is least visible on.

**MSFT is where the damage actually shows.** It took three of the six nulls and trades around £468, more than double the overall mean. Filling those three gaps with £226.46 drags MSFT's mean price down by **£65.88 — a 14% error** on an instrument whose price was never remotely near that number. JNJ is wrong in the opposite direction, pushed up £13.47 from £152.

**Why per-instrument is the defensible choice.** A global mean assumes every instrument is drawn from one distribution. These aren't — MSFT trades near £468, XOM near £118. The overall mean is a number no instrument in the basket actually trades at, so filling with it doesn't approximate any real price. Grouping by ticker fills each gap from that instrument's own price level, which is at least the right order of magnitude.

**A caveat worth recording even so.** A group mean uses *future* data to fill a past gap — MSFT's mean includes June trades, so a January null gets filled with information that didn't exist in January. For exploratory analysis that's acceptable. For a backtest or any "what did we know on 3 March" regulatory question it's lookahead bias, and it silently flatters results. A forward fill (carry the last observed price) would only use information already available at that point in time. This is Week 15's point-in-time correctness, and it's the reason this fill is fine *here* and would not be fine in a production price series.

### The 3 missing volumes: flagged, not filled

**Volume is deliberately left unfilled.** Price has a defensible "roughly what was this instrument worth around then" answer; volume does not. A trade was either 2,500 shares or it wasn't — inventing a number manufactures an event that never happened. Volume also feeds straight into notional value (price × volume), VWAP and position sizing, so a fabricated volume becomes fabricated money downstream, and nothing marks it as invented.

A mean is doubly wrong here: it produces fractional shares, which aren't a thing.

What would be done in production, in order of preference:

1. **Go back to source.** Missing volume is usually an ingestion failure rather than a data failure — re-request the day from the vendor or reload the partition.
2. **Quarantine.** Route affected rows to a rejects table, let the clean rows flow, and alert someone. Nothing is silently dropped and nothing is silently invented.
3. **Flag and keep.** Where downstream genuinely needs the trade to exist, retain it with an explicit marker so consumers can exclude it from volume-weighted calculations.

Option 3 is taken below, since this is an analysis notebook rather than a pipeline. `volume` is also converted to pandas' nullable `Int64`, which restores integer semantics while still representing the missing values — resolving the float upcast noted on Monday.

In [22]:
# flag the missing volumes rather than inventing values for them
raw_trades['volume_is_missing'] = raw_trades['volume'].isna()

# nullable Int64 keeps whole-share semantics while still holding the missing values
raw_trades['volume'] = raw_trades['volume'].astype('Int64')

print(raw_trades.dtypes, '\n')
print(f"rows flagged: {raw_trades['volume_is_missing'].sum()}\n")
raw_trades[raw_trades['volume_is_missing']]

trade_id               int64
ticker                   str
price                float64
volume                 Int64
venue_code               str
side                     str
volume_is_missing       bool
dtype: object 

rows flagged: 3



,trade_id,ticker,price,volume,venue_code,side,volume_is_missing
trade_date,,,,,,,
2026-01-27,1034,XOM,124.22,<NA>,XNYS,BUY,True
2026-03-13,1005,AAPL,209.18,<NA>,XNAS,SELL,True
2026-03-18,1038,XOM,115.35,<NA>,XNYS,BUY,True


In [23]:
raw_trades.groupby(['ticker'])['price'].mean()

ticker
AAPL    221.882000
JNJ     152.345556
JPM     214.339091
MSFT    467.998750
XOM     117.824000
Name: price, dtype: float64

In [24]:
raw_trades.groupby(["ticker"]).agg({'price': 'mean', 'volume': 'sum'})

,price,volume
ticker,,
AAPL,221.882000,32250
JNJ,152.345556,54250
JPM,214.339091,16000
MSFT,467.998750,63000
XOM,117.824000,14750
